**Train Emotion Datasets(YOLO FORMAT AffectNet + RAF-DB + FERPlus)**

In [ ]:
!pip install torch torchvision torchaudio timm --quiet
!pip install onnx onnxruntime --quiet
!pip install pandas numpy matplotlib scikit-learn --quiet

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import cv2
from google.colab import drive

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!unzip "/content/drive/MyDrive/EmotionDatasets/Raw/YOLO_format.zip" -d YOLO_format


In [ ]:
!find YOLO_format -maxdepth 3 -type d

In [ ]:
import os
import pandas as pd
import shutil

# drive directory for YOLO_AffectNet:
base_drive = "/content/drive/MyDrive/EmotionDatasets/YOLO_AffectNet"

emotion_map = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 6:6}

rows = []

for split in ["train", "valid", "test"]:
    img_dir = f"YOLO_format/YOLO_format/{split}/images"
    label_dir = f"YOLO_format/YOLO_format/{split}/labels"

    print(f"Processing split: {split}")
    print(f"Images from: {img_dir}")
    print(f"Labels from: {label_dir}")
    print("-"*50)

    for filename in os.listdir(img_dir):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):

            img_path = os.path.join(img_dir, filename)

            # label file path
            file_stem = filename.rsplit(".", 1)[0]
            label_file = os.path.join(label_dir, file_stem + ".txt")

            if not os.path.exists(label_file):
                continue

            # read class id
            with open(label_file, "r") as f:
                cls_id = int(f.read().split()[0])

            # new path inside Drive
            new_path = f"images/{split}_{filename}"

            # copy image to Drive folder
            shutil.copy(img_path, f"{base_drive}/{new_path}")

            # save row
            rows.append([new_path, cls_id, split])

df = pd.DataFrame(rows, columns=["path","expression","split"])
df.to_csv(f"{base_drive}/labels.csv", index=False)

print("\nYOLO AffectNet preprocessing complete!")
print(f"Total samples processed: {len(df)}")
df.head()


In [ ]:
!rm -rf "/content/drive/MyDrive/EmotionDatasets/RAF_DB/images"
!rm "/content/drive/MyDrive/EmotionDatasets/RAF_DB/labels.csv"


In [ ]:
!unzip "/content/drive/MyDrive/EmotionDatasets/Raw/RAF_DB.zip" -d RAF_DB

In [ ]:
!find RAF_DB -maxdepth 5 -type d

In [ ]:
import os
import pandas as pd
import shutil

base_drive = "/content/drive/MyDrive/EmotionDatasets/RAF_DB"
os.makedirs(f"{base_drive}/images", exist_ok=True)

emotion_map = {
    1: 0,
    2: 1,
    3: 2,
    4: 3,
    5: 4,
    6: 5,
    7: 6
}

rows = []

for split in ["train", "test"]:
    split_dir = f"RAF_DB/RAF_DB/DATASET/{split}"

    print("Processing:", split_dir)

    for cls_folder in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls_folder)

        if not os.path.isdir(cls_path):
            continue

        cls_id = int(cls_folder)
        mapped = emotion_map[cls_id]

        for img_name in os.listdir(cls_path):
            if img_name.lower().endswith((".jpg",".png",".jpeg")):

                src = os.path.join(cls_path, img_name)

                new_path = f"images/{img_name}"

                shutil.copy(src, f"{base_drive}/{new_path}")

                rows.append([new_path, mapped, split])

df = pd.DataFrame(rows, columns=["path","expression","split"])
df.to_csv(f"{base_drive}/labels.csv", index=False)

print("Total samples:", len(df))

In [ ]:
!ls -lh "/content/drive/MyDrive/EmotionDatasets/RAF_DB"

In [ ]:
!unzip "/content/drive/MyDrive/EmotionDatasets/Raw/FERPlus.zip" -d FERPlus


In [ ]:
!find FERPlus -maxdepth 3 -type d

In [ ]:
import os
import pandas as pd
import shutil

print("Starting FERPlus preprocessing...\n")

base_drive = "/content/drive/MyDrive/EmotionDatasets/FERPlus"
os.makedirs(f"{base_drive}/images", exist_ok=True)

emotion_map = {
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "happy": 3,
    "sad": 4,
    "surprise": 5,
    "neutral": 6,
    "contempt": 1
}

rows = []
splits = ["train", "validation", "test"]

for split in splits:
    split_dir = f"FERPlus/FERPlus/{split}"
    if not os.path.exists(split_dir):
        split_dir = f"FERPlus/{split}"

    print(f"Processing split: {split}")
    print(f"Directory: {split_dir}")
    print("-" * 50)

    for folder in os.listdir(split_dir):
        folder_path = os.path.join(split_dir, folder)
        if not os.path.isdir(folder_path):
            continue

        f_lower = folder.lower()
        if f_lower not in emotion_map:
            print(f"Skipping {folder}")
            continue

        label_id = emotion_map[f_lower]

        for img_name in os.listdir(folder_path):
            if not img_name.lower().endswith((".jpg",".jpeg",".png")):
                continue

            src_path = os.path.join(folder_path, img_name)
            new_path = f"images/{split}_{folder}_{img_name}"

            shutil.copy(src_path, f"{base_drive}/{new_path}")
            rows.append([new_path, label_id, split])

df = pd.DataFrame(rows, columns=["path","expression","split"])
df.to_csv(f"{base_drive}/labels.csv", index=False)

print("\n FERPlus preprocessing complete!")
print("Total images processed:", len(df))
df.head()


In [ ]:
!ls -lh "/content/drive/MyDrive/EmotionDatasets/FERPlus"

In [ ]:
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/EmotionDatasets"

paths = {
    "YOLO_AffectNet": f"{BASE}/YOLO_AffectNet/labels.csv",
    "RAF_DB": f"{BASE}/RAF_DB/labels.csv",
    "FERPlus": f"{BASE}/FERPlus/labels.csv",
}

dfs = []
for key, p in paths.items():
    df = pd.read_csv(p)
    df["dataset"] = key
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
len(df_all), df_all.head()


**fix full path globally**

In [ ]:
def fix_path(row):
    return f"{row['dataset']}/" + row['path']

df_all["full_path"] = df_all.apply(fix_path, axis=1)
df_all.head()


In [ ]:
#Remap YOLO AffectNet Label 7 → 1 (Disgust) it will prevent the CUDA crash
df_all.loc[df_all["expression"] == 7, "expression"] = 1

# Final check (no label 7 should remain)
print(df_all["expression"].value_counts())
print("Min/Max:", df_all["expression"].min(), df_all["expression"].max())


In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train_df, val_df = train_test_split(
    df_all,
    test_size=0.15,
    stratify=df_all["expression"],
    random_state=42
)

len(train_df), len(val_df)


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_t = T.Compose([
    T.ToPILImage(),
    T.Resize((256,256)),
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2,0.2,0.2,0.1),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_t = T.Compose([
    T.ToPILImage(),
    T.Resize((224,224)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FERDataset(Dataset):
    def __init__(self, df, base, transform):
        self.df = df.reset_index(drop=True)
        self.base = base
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.base, row["full_path"])
        label = int(row["expression"])

        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(img_path)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(img), label


In [ ]:
train_ds = FERDataset(train_df, BASE, train_t)
val_ds   = FERDataset(val_df, BASE, val_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

len(train_loader), len(val_loader)


In [ ]:
import timm

class EmotionModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.model = timm.create_model("efficientnet_b2", pretrained=True)
        in_features = self.model.classifier.in_features
        self.model.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

model = EmotionModel().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()


In [ ]:
# Enable AMP for Faster Training
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()


In [ ]:
#AMP-Based training functions optimized edition
def train_epoch_amp(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()

        with autocast():  # Mixed precision
            out = model(imgs)
            loss = criterion(out, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * imgs.size(0)
        preds = out.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

    return total_loss/total, correct/total


@torch.no_grad()
def validate_amp(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        with autocast():
            out = model(imgs)
            loss = criterion(out, labels)

        total_loss += loss.item() * imgs.size(0)
        preds = out.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)

    return total_loss/total, correct/total


In [ ]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += imgs.size(0)

    return total_loss/total_samples, total_correct/total_samples


@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += imgs.size(0)

    return total_loss/total_samples, total_correct/total_samples


In [ ]:
best_acc = 0
EPOCHS = 12

print("Training from epoch 1 to 12")

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc = train_epoch_amp(model, train_loader)
    va_loss, va_acc = validate_amp(model, val_loader)

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f" Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc:.4f}")
    print(f" Val Loss:   {va_loss:.4f} | Val Acc:   {va_acc:.4f}")

    if va_acc > best_acc:
        best_acc = va_acc
        # Save to DRIVE so checkpoint survives disconnects
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_combined_model.pth")
        print("model saved.")


In [ ]:
# put model in inference mode
import torch
model.eval()


In [ ]:
!pip install onnx onnxscript
!pip install onnxruntime


In [ ]:
import torch

# dummy input to match preprocessing size
dummy = torch.randn(1, 3, 224, 224).to(device)

#save to Google Drive
onnx_path = "/content/drive/MyDrive/combined_fer_model.onnx"

# Export to ONNX
with torch.inference_mode():
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=["input"],
        output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}},
        opset_version=17  # modern opset
    )

print("ONNX model exported to:", onnx_path)


In [ ]:
from google.colab import files
files.download(onnx_path)